# build_html.ipynb

Converts `report.qmd` into a fully self-contained `report.html` with embedded images,
a sticky table of contents, and a client-facing visual design.

## How this notebook works

Each step is a self-contained cell pair: a markdown cell explaining the goal,
followed by a code cell doing exactly one thing. Run top to bottom to regenerate
the report from scratch.

## When to re-run

- After editing `report.qmd` (content changes)
- After re-running any model notebook (new figures)
- After changing the CSS or hero section below

## Output

A single self-contained `report.html` at the project root.
All figures are embedded as base64, no external dependencies,
no broken image links, works offline and as an email attachment.

## Step 1: Configuration

All paths and settings in one place.
If the project moves, only this cell needs updating.

In [55]:
import os
import re
import base64
from datetime import date

try:
    import markdown as md_lib
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'markdown', '-q'])
    import markdown as md_lib

# ── Paths ─────────────────────────────────────────────────────────────────────
QMD_PATH  = '../report.qmd'
HTML_PATH = '../report.html'
FIG_ROOT  = os.path.dirname(os.path.abspath(QMD_PATH))

# ── Project metadata ──────────────────────────────────────────────────────────
AUTHORS = 'Paula Barghout · Elena Fuchs · Tamara Marcet'
COURSE  = 'HSLU, Applied Machine Learning and Predictive Modelling 1, FS26'
CLIENT  = 'City of Zurich · EWZ Grid · 2022–2025'

# ── Key metrics for the metrics bar ──────────────────────────────────────────
METRICS = [
    ('35,000+', 'Hourly observations'),
    ('6',       'ML models compared'),
    ('881 kW',  'Best forecast RMSE'),
    ('1.2%',    'Of mean demand'),
    ('0.96',    'Peak hour AUC'),
    ('99.7%',   'Variance explained (SVM)'),
]

print('Configuration loaded ✓')

Configuration loaded ✓


## Step 2: Load & Parse QMD

Load `report.qmd` and strip the YAML front matter, leaving only
the markdown body that will become the HTML content.

In [56]:
# ── Load QMD ──────────────────────────────────────────────────────────────────
with open(QMD_PATH, encoding='utf-8') as f:
    qmd = f.read()

# ── Strip YAML front matter (everything between the first two ---) ─────────────
body = re.sub(r'^---.*?---\s*', '', qmd, flags=re.DOTALL)

print(f'QMD loaded         ✓  ({len(qmd):,} characters)')
print(f'Body after YAML    ✓  ({len(body):,} characters)')

QMD loaded         ✓  (33,103 characters)
Body after YAML    ✓  (32,747 characters)


## Step 3: Embed Images as Base64

All figures are read from disk and embedded directly into the HTML
as base64 strings. This makes the report fully self-contained, no broken links, works offline, safe to email or submit.

Missing figures are flagged but do not stop the build.

In [57]:
embedded = []
missing  = []

def embed_image(m):
    rel_path = m.group(1)
    full_path = os.path.join(FIG_ROOT, rel_path)
    if not os.path.exists(full_path):
        missing.append(rel_path)
        return m.group(0)
    with open(full_path, 'rb') as f:
        data = base64.b64encode(f.read()).decode()
    embedded.append(rel_path)
    return f'![](data:image/png;base64,{data})'

body = re.sub(r'!\[\]\(([^)]+\.png)\)', embed_image, body)

print(f'Images embedded    ✓  ({len(embedded)})')
print(f'Images missing     {"⚠️" if missing else "✓"} ({len(missing)})')
for p in missing:
    print(f'   ⚠️  {p}')

Images embedded    ✓  (19)
Images missing     ✓ (0)


## Step 4: CSS Design System

The visual design uses a sustainable green palette chosen to feel appropriate
for a City of Zurich energy report, professional, clean, and trustworthy.

**Palette**
- Forest green `#1a4a2e` — headings, hero background
- Sage `#4a7c59` — accents, TOC, subheadings, roadmap line
- Cream `#f7f4ee` — page background
- Amber `#c8861a` — limitations cards, alert accents
- Moss `#7a9e87` — muted labels, TOC links

**Typography**
- Playfair Display — headings and pull quotes (editorial, authoritative)
- Source Serif 4 — body text and subheadings (readable, warm)
- SFMono-Regular — code blocks

**Component inventory**
- Hero + wind turbine illustration
- Metrics bar (6 key numbers)
- Sticky sidebar TOC
- Pull quote blockquotes with leave accent
- Lead author tags
- Roadmap timeline (recommendations)
- Amber alert cards (limitations)
- AI usage blocks (good / hard / careful)
- Code blocks with dark background

To change the look: edit the CSS variables in `:root` below.
Everything else inherits from them automatically.

In [58]:
css = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;600;700&family=Source+Serif+4:wght@300;400;600&display=swap');

/* ═══════════════════════════════════════════════════════════════════════════
   DESIGN TOKENS
   ═══════════════════════════════════════════════════════════════════════════ */
:root {
    --forest:      #1a4a2e;
    --sage:        #4a7c59;
    --sage-light:  #e8f0e8;
    --cream:       #f7f4ee;
    --amber:       #c8861a;
    --amber-light: #fdf3e0;
    --body:        #2d3d35;
    --border:      #c8d8c8;
    --moss:        #7a9e87;
    --white:       #ffffff;
}

/* ═══════════════════════════════════════════════════════════════════════════
   RESET & BASE
   ═══════════════════════════════════════════════════════════════════════════ */
* { box-sizing: border-box; margin: 0; padding: 0; }

body {
    font-family: 'Source Serif 4', Georgia, serif;
    font-size: 16px;
    line-height: 1.8;
    color: var(--body);
    background: var(--cream);
}

/* ═══════════════════════════════════════════════════════════════════════════
   HERO
   ═══════════════════════════════════════════════════════════════════════════ */
#hero {
    background: linear-gradient(150deg, var(--forest) 0%, #2d6b45 60%, #3d8a5a 100%);
    color: var(--white);
    padding: 4rem 3rem 5rem;
    position: relative;
    overflow: hidden;
}
#hero::before {
    content: '';
    position: absolute;
    bottom: -2px; left: 0; right: 0;
    height: 60px;
    background: var(--cream);
    clip-path: ellipse(55% 100% at 50% 100%);
}
#hero .hero-inner { position: relative; z-index: 1; max-width: 700px; }
#hero .eyebrow {
    font-size: 0.78rem; font-weight: 600; letter-spacing: .14em;
    text-transform: uppercase; color: rgba(255,255,255,0.55); margin-bottom: 1.2rem;
}
#hero h1 {
    font-family: 'Playfair Display', serif;
    font-size: 2.8rem; line-height: 1.15; font-weight: 700;
    color: var(--white); margin-bottom: 1.4rem;
}
#hero .lead {
    font-size: 1.05rem; color: rgba(255,255,255,0.8);
    line-height: 1.75; margin-bottom: 2.5rem; max-width: 580px;
}
#hero .meta {
    font-size: 0.82rem; color: rgba(255,255,255,0.45);
    border-top: 1px solid rgba(255,255,255,0.12);
    padding-top: 1.2rem; display: flex; gap: 2rem; flex-wrap: wrap;
}
#hero .illustration {
    position: absolute; right: 3rem; top: 50%;
    transform: translateY(-50%); opacity: 0.12; z-index: 0;
}

/* ═══════════════════════════════════════════════════════════════════════════
   METRICS BAR
   ═══════════════════════════════════════════════════════════════════════════ */
#metrics-bar {
    background: var(--sage);
    display: flex; justify-content: center; flex-wrap: wrap;
}
.metric-card {
    padding: 1.5rem 2.5rem; text-align: center;
    border-right: 1px solid rgba(255,255,255,0.15); color: var(--white);
}
.metric-card:last-child { border-right: none; }
.metric-value {
    font-family: 'Playfair Display', serif;
    font-size: 2.1rem; line-height: 1; display: block;
}
.metric-label {
    font-size: 0.72rem; color: rgba(255,255,255,0.6);
    text-transform: uppercase; letter-spacing: .09em;
    margin-top: .4rem; display: block;
}

/* ═══════════════════════════════════════════════════════════════════════════
   PAGE LAYOUT
   ═══════════════════════════════════════════════════════════════════════════ */
#layout {
    display: flex; max-width: 1200px;
    margin: 0 auto; padding: 1rem 1.5rem 3rem; gap: 3rem;
}

/* ═══════════════════════════════════════════════════════════════════════════
   SIDEBAR TOC
   ═══════════════════════════════════════════════════════════════════════════ */
#toc {
    flex: 0 0 200px; position: sticky; top: 2rem;
    align-self: flex-start; font-size: 0.82rem;
    border-right: 2px solid var(--border);
    padding-right: 1.5rem; max-height: 90vh; overflow-y: auto;
}
#toc h2 {
    font-size: 0.7rem; text-transform: uppercase; letter-spacing: .12em;
    color: var(--sage); margin-bottom: .9rem; font-weight: 600;
    border-bottom: none; padding-bottom: 0;
}
#toc ul { list-style: none; }
#toc li { margin: .4rem 0; }
#toc li.toc-sub { padding-left: 1rem; }
#toc a { color: var(--moss); text-decoration: none; transition: color .2s; }
#toc a:hover { color: var(--forest); }

/* ═══════════════════════════════════════════════════════════════════════════
   CONTENT TYPOGRAPHY
   ═══════════════════════════════════════════════════════════════════════════ */
#content { flex: 1; min-width: 0; }

h2 {
    font-family: 'Playfair Display', serif;
    font-size: 1.7rem; font-weight: 700; color: var(--forest);
    margin: 3rem 0 .9rem; padding-bottom: .5rem;
    border-bottom: 2px solid var(--border);
}
h3 {
    font-family: 'Source Serif 4', serif;
    font-size: 1.1rem; font-weight: 600; color: var(--sage);
    margin: 2rem 0 .6rem;
}
h4 {
    font-family: 'Source Serif 4', serif;
    font-size: 0.95rem; font-weight: 600; color: var(--forest);
    margin: 1.6rem 0 0.3rem;
}
p { margin: .9rem 0; }
strong { color: var(--forest); }
em { color: var(--moss); }
ul, ol { margin: .7rem 0 .7rem 1.6rem; }
li { margin: .4rem 0; }

/* ═══════════════════════════════════════════════════════════════════════════
   PULL QUOTES
   ═══════════════════════════════════════════════════════════════════════════ */
blockquote {
    position: relative; margin: 2.5rem 0;
    padding: 1.5rem 1.8rem 1.5rem 2rem;
    border-top: 3px solid var(--sage);
    border-bottom: 3px solid var(--sage);
}
blockquote::before {
    content: '🌿'; position: absolute; top: -0.9rem; left: 1.5rem;
    font-size: 1.2rem; background: var(--cream); padding: 0 0.4rem;
}
blockquote p {
    font-family: 'Playfair Display', serif;
    font-size: 1.25rem; font-weight: 700;
    line-height: 1.45; color: var(--forest); margin: 0;
}
blockquote strong { color: var(--forest); }

/* ═══════════════════════════════════════════════════════════════════════════
   LEAD AUTHOR TAG
   ═══════════════════════════════════════════════════════════════════════════ */
.lead-tag {
    display: inline-block; font-size: 0.72rem; font-weight: 600;
    letter-spacing: .07em; text-transform: uppercase;
    color: var(--sage); background: var(--sage-light);
    border: 1px solid var(--border);
    padding: .18rem .7rem; border-radius: 20px; margin-bottom: 1rem;
}

/* ═══════════════════════════════════════════════════════════════════════════
   IMAGES
   ═══════════════════════════════════════════════════════════════════════════ */
img {
    max-width: 100%; height: auto; display: block;
    margin: 1.8rem auto; border-radius: 10px;
    box-shadow: 0 3px 20px rgba(26,74,46,0.1);
}

/* ═══════════════════════════════════════════════════════════════════════════
   TABLES
   ═══════════════════════════════════════════════════════════════════════════ */
table {
    border-collapse: collapse; width: 100%; margin: 1.4rem 0;
    font-size: 0.9rem; border-radius: 10px; overflow: hidden;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
}
th {
    background: var(--forest); color: var(--white);
    padding: .7rem 1rem; text-align: left;
    font-weight: 600; font-size: 0.82rem; letter-spacing: .04em;
}
td { padding: .55rem 1rem; border-bottom: 1px solid var(--border); }
tr:nth-child(even) td { background: var(--sage-light); }
tr:last-child td { border-bottom: none; }

/* ═══════════════════════════════════════════════════════════════════════════
   CODE BLOCKS
   ═══════════════════════════════════════════════════════════════════════════ */
code {
    background: var(--sage-light); padding: .15em .4em; border-radius: 3px;
    font-family: 'SFMono-Regular', Consolas, monospace;
    font-size: .85em; color: var(--sage);
}
.code-block {
    margin: 1rem 0 1.8rem; border: 1px solid var(--border);
    border-radius: 10px; overflow: hidden;
}
.code-label {
    background: var(--sage-light); color: var(--sage);
    font-size: 0.73rem; font-weight: 700; letter-spacing: .08em;
    text-transform: uppercase; padding: .38rem 1rem;
    border-bottom: 1px solid var(--border);
}
.code-block pre { margin: 0; padding: 1rem 1.2rem; background: #0d1f13; overflow-x: auto; }
.code-block pre code { background: none; color: #a8d8a8; font-size: 0.83rem; padding: 0; border-radius: 0; }

/* ═══════════════════════════════════════════════════════════════════════════
   RECOMMENDATIONS — ROADMAP TIMELINE
   ═══════════════════════════════════════════════════════════════════════════ */
.roadmap { position: relative; margin: 1.5rem 0 2.5rem; padding-left: 2.5rem; }
.roadmap::before {
    content: ''; position: absolute;
    left: 0.75rem; top: 1.2rem; bottom: 1.2rem;
    width: 2px;
    background: linear-gradient(to bottom, var(--sage), var(--moss), var(--border));
}
.roadmap-step {
    position: relative; margin-bottom: 2rem;
    background: var(--white); border: 1px solid var(--border);
    border-radius: 10px; padding: 1.3rem 1.5rem;
}
.roadmap-step:last-child { margin-bottom: 0; }
.roadmap-step::before {
    content: ''; position: absolute;
    left: -1.85rem; top: 1.4rem;
    width: 12px; height: 12px; border-radius: 50%;
    background: var(--sage); border: 2px solid var(--white);
    box-shadow: 0 0 0 2px var(--sage);
}
.roadmap-step::after {
    content: ''; position: absolute;
    left: -1.18rem; top: 1.85rem;
    width: 0.75rem; height: 2px; background: var(--sage);
}
.roadmap-header { display: flex; align-items: center; gap: 0.8rem; margin-bottom: 0.5rem; }
.roadmap-number {
    font-family: 'Playfair Display', serif;
    font-size: 1.6rem; font-weight: 700;
    color: var(--border); line-height: 1; min-width: 1.8rem;
}
.roadmap-meta { display: flex; flex-direction: column; gap: 0.1rem; }
.roadmap-priority {
    font-size: 0.65rem; font-weight: 700; letter-spacing: .12em;
    text-transform: uppercase; color: var(--moss);
}
.roadmap-title {
    font-family: 'Playfair Display', serif;
    font-size: 1.05rem; font-weight: 700; color: var(--forest);
}
.roadmap-detail {
    font-size: 0.88rem; color: var(--body); line-height: 1.65;
    border-top: 1px solid var(--border); padding-top: 0.6rem; margin-top: 0.6rem;
}

/* ═══════════════════════════════════════════════════════════════════════════
   LIMITATIONS — GREEN ENERGY CARDS
   ═══════════════════════════════════════════════════════════════════════════ */
.limitations {
    display: grid; grid-template-columns: repeat(2, 1fr);
    gap: 1rem; margin: 1.5rem 0 2rem;
}
.lim-card {
    background: var(--white); border: 1px solid var(--border);
    border-radius: 12px; padding: 1.4rem 1.5rem;
    position: relative; overflow: hidden;
}
.lim-card::before {
    content: ''; position: absolute; top: 0; left: 0; right: 0;
    height: 4px;
    background: linear-gradient(90deg, var(--forest), var(--sage), var(--moss));
}
.lim-card::after {
    content: ''; position: absolute;
    bottom: -30px; right: -30px;
    width: 100px; height: 100px; border-radius: 50%;
    background: var(--sage-light); opacity: 0.5;
}
.lim-icon { font-size: 1.6rem; margin-bottom: .5rem; display: block; position: relative; z-index: 1; }
.lim-title {
    font-family: 'Playfair Display', serif; font-size: 1rem; font-weight: 700;
    color: var(--forest); margin-bottom: .4rem; position: relative; z-index: 1;
}
.lim-text { font-size: 0.85rem; color: var(--body); line-height: 1.65; position: relative; z-index: 1; }

/* ═══════════════════════════════════════════════════════════════════════════
   AI USAGE — GREEN SIGNAL BARS
   ═══════════════════════════════════════════════════════════════════════════ */
.ai-section { margin: 1.5rem 0 2rem; display: flex; flex-direction: column; gap: 0.8rem; }
.ai-block {
    display: grid; grid-template-columns: 5px 56px 1fr;
    align-items: stretch; border-radius: 12px;
    overflow: hidden; border: 1px solid var(--border);
    background: var(--white);
}
.ai-signal { width: 5px; align-self: stretch; }
.ai-signal.good    { background: var(--forest); }
.ai-signal.hard    { background: var(--sage); }
.ai-signal.careful { background: var(--moss); }
.ai-icon {
    display: flex; align-items: center; justify-content: center;
    padding: 1.2rem 0; align-self: stretch;
}
.ai-icon.good    { background: var(--sage-light); }
.ai-icon.hard    { background: #d8ece0; }
.ai-icon.careful { background: #eaf4ef; }
.ai-icon svg { width: 20px; height: 20px; }
.ai-content { padding: 1rem 1.2rem; }
.ai-label {
    font-size: 0.68rem; font-weight: 700; letter-spacing: .1em;
    text-transform: uppercase; color: var(--moss); margin-bottom: .3rem;
}
.ai-text { font-size: 0.9rem; color: var(--body); line-height: 1.7; }

/* ═══════════════════════════════════════════════════════════════════════════
   END ILLUSTRATION — WIND TURBINE FOOTER
   ═══════════════════════════════════════════════════════════════════════════ */
.page-footer {
    margin-top: 4rem;
    background: linear-gradient(180deg, var(--cream) 0%, var(--sage-light) 40%, var(--sage) 100%);
    border-radius: 16px; overflow: hidden;
    padding: 2rem 2rem 0;
    text-align: center;
}
.page-footer .footer-text {
    font-family: 'Playfair Display', serif;
    font-size: 1.1rem; color: var(--forest);
    margin-bottom: 1.5rem; font-style: italic;
}

/* ═══════════════════════════════════════════════════════════════════════════
   RESPONSIVE
   ═══════════════════════════════════════════════════════════════════════════ */
@media (max-width: 768px) {
    #hero { padding: 2.5rem 1.5rem 3rem; }
    #hero h1 { font-size: 1.9rem; }
    #hero .illustration { display: none; }
    #layout { flex-direction: column; padding: 1rem; }
    #toc { position: static; border-right: none; border-bottom: 2px solid var(--border); padding-bottom: 1rem; max-height: none; }
    .metric-card { padding: 1rem 1.5rem; }
    .limitations { grid-template-columns: 1fr; }
}

/* ═══════════════════════════════════════════════════════════════════════════
   PRINT
   ═══════════════════════════════════════════════════════════════════════════ */
@media print {
    #hero::before, #metrics-bar { display: none; }
    #layout { flex-direction: column; gap: 0; padding: 0; }
    #toc { position: static; border-right: none; border-bottom: 1px solid #ccc; max-height: none; padding: 0 0 1rem 0; margin-bottom: 1.5rem; }
    body { font-size: 11px; background: white; }
    h2 { font-size: 1.2rem; }
    h3 { font-size: 1rem; }
    img { max-width: 85%; page-break-inside: avoid; }
}
"""

print('CSS defined ✓')

CSS defined ✓


## Step 5: Hero & Metrics Bar

The hero is the first thing the client sees, it sets the tone.
It includes a wind turbine SVG illustration, a strong headline,
a one-paragraph summary, and author/course metadata.

The metrics bar sits directly below and gives the reader
the six most important numbers at a glance before they
read a single word of the report.

To update a metric: edit the METRICS list in Cell 2.

In [59]:
# ── Wind turbine SVG illustration ─────────────────────────────────────────────
illustration = """
<svg class="illustration" width="320" height="400" viewBox="0 0 320 400"
     fill="none" xmlns="http://www.w3.org/2000/svg">
  <polygon points="155,380 165,380 162,180 158,180" fill="white"/>
  <circle cx="160" cy="175" r="8" fill="white"/>
  <ellipse cx="160" cy="120" rx="8" ry="55" fill="white"
           transform="rotate(0 160 175)"/>
  <ellipse cx="160" cy="120" rx="8" ry="55" fill="white"
           transform="rotate(120 160 175)"/>
  <ellipse cx="160" cy="120" rx="8" ry="55" fill="white"
           transform="rotate(240 160 175)"/>
  <circle cx="270" cy="60" r="22" fill="white" opacity="0.6"/>
  <line x1="270" y1="20" x2="270" y2="10" stroke="white" stroke-width="3" opacity="0.5"/>
  <line x1="270" y1="100" x2="270" y2="110" stroke="white" stroke-width="3" opacity="0.5"/>
  <line x1="230" y1="60" x2="220" y2="60" stroke="white" stroke-width="3" opacity="0.5"/>
  <line x1="310" y1="60" x2="320" y2="60" stroke="white" stroke-width="3" opacity="0.5"/>
  <line x1="242" y1="32" x2="235" y2="25" stroke="white" stroke-width="3" opacity="0.5"/>
  <line x1="298" y1="88" x2="305" y2="95" stroke="white" stroke-width="3" opacity="0.5"/>
  <line x1="298" y1="32" x2="305" y2="25" stroke="white" stroke-width="3" opacity="0.5"/>
  <line x1="242" y1="88" x2="235" y2="95" stroke="white" stroke-width="3" opacity="0.5"/>
  <ellipse cx="160" cy="390" rx="200" ry="30" fill="white" opacity="0.08"/>
  <ellipse cx="60"  cy="395" rx="100" ry="20" fill="white" opacity="0.06"/>
  <ellipse cx="270" cy="395" rx="80"  ry="18" fill="white" opacity="0.06"/>
</svg>
"""

# ── Hero block ─────────────────────────────────────────────────────────────────
hero = f"""
<div id="hero">
  {illustration}
  <div class="hero-inner">
    <div class="eyebrow">{CLIENT}</div>
    <h1>When does Zurich need power, <br>and can we predict it?</h1>
    <p class="lead">
      Every hour, EWZ supplies electricity to 400,000 residents and thousands of
      businesses across Zurich. Getting demand forecasts wrong, even by a few
      percent, means emergency procurement costs or wasted capacity. This report
      shows how machine learning can make those forecasts reliable, actionable,
      and transparent.
    </p>
    <div class="meta">
      <span>{AUTHORS}</span>
      <span>{COURSE}</span>
      <span>{date.today().strftime('%B %d, %Y')}</span>
    </div>
  </div>
</div>
"""

# ── Metrics bar ────────────────────────────────────────────────────────────────
metric_cards = '\n'.join(
    f"""  <div class="metric-card">
    <span class="metric-value">{value}</span>
    <span class="metric-label">{label}</span>
  </div>"""
    for value, label in METRICS
)
metrics_bar = f'<div id="metrics-bar">\n{metric_cards}\n</div>'

print('Hero built         ✓')
print('Metrics bar built  ✓')

Hero built         ✓
Metrics bar built  ✓


## Step 6: Process Code Blocks

The QMD contains `<details>` blocks with collapsible code snippets.
These need to be extracted before markdown conversion, transformed into
styled HTML code blocks, then re-injected after conversion.

We stash them with a unique placeholder key to survive the markdown parser.

In [60]:
details_blocks = {}

def stash_details(m):
    key   = f'CODEBLOCK_{len(details_blocks)}_END'
    inner = m.group(1)
    code_m = re.search(r'```(?:python)?\n(.*?)```', inner, re.DOTALL)
    code_content = code_m.group(1).rstrip() if code_m else inner.strip()

    def esc(s):
        return s.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')

    html = (
        '<div class="code-block">'
        '<div class="code-label">Python</div>'
        f'<pre><code>{esc(code_content)}</code></pre>'
        '</div>'
    )
    details_blocks[key] = html
    return f'\n\n{key}\n\n'

body = re.sub(r'<details>(.*?)</details>', stash_details, body, flags=re.DOTALL)

print(f'Code blocks stashed ✓  ({len(details_blocks)})')

Code blocks stashed ✓  (0)


## Step 7: Table of Contents & Heading Anchors

Auto-generates a sticky sidebar TOC from all `##` and `###` headings.
Each heading gets a unique anchor ID so TOC links scroll to the right place.

In [61]:
toc_items = []
h2_count  = [0]
h3_count  = [0]

def make_anchor(text):
    return re.sub(r'[^a-z0-9-]', '', text.lower().replace(' ', '-'))

def add_heading_id(m):
    hashes = m.group(1)
    text   = re.sub(r'[*_`]', '', m.group(2).strip())
    anchor = make_anchor(text)
    if len(hashes) == 2:
        h2_count[0] += 1
        h3_count[0]  = 0
        num = f'{h2_count[0]}. '
        toc_items.append(f'<li><a href="#{anchor}">{num}{text}</a></li>')
        return f'<h2 id="{anchor}">{num}{text}</h2>'
    elif len(hashes) == 3:
        h3_count[0] += 1
        num = f'{h2_count[0]}.{h3_count[0]}. '
        toc_items.append(f'<li class="toc-sub"><a href="#{anchor}">{num}{text}</a></li>')
        return f'<h3 id="{anchor}">{num}{text}</h3>'
    else:
        return f'<h4>{text}</h4>'

body_tagged = re.sub(r'^(#{2,4})\s+(.+)$', add_heading_id, body, flags=re.MULTILINE)
html_body   = md_lib.markdown(body_tagged, extensions=['tables', 'fenced_code'])

for key, html in details_blocks.items():
    html_body = html_body.replace(f'<p>{key}</p>', html).replace(key, html)

toc_html = (
    '<nav id="toc">'
    '<h2>Contents</h2>'
    '<ul>' + ''.join(toc_items) + '</ul>'
    '</nav>'
)

print(f'TOC generated      ✓  ({len(toc_items)} entries)')
print(f'Headings anchored  ✓')

TOC generated      ✓  (23 entries)
Headings anchored  ✓


## Step 8: Inject Visual Components

Three sections of the report are replaced with styled HTML components
that go beyond what markdown can express:

1. **Operational Recommendations**: a roadmap timeline with a connecting
   vertical line, numbered steps, priority labels, and detail text.
   Edit the `RECOMMENDATIONS` list to change content.

2. **Limitations**: amber alert cards with left border accent and icons.
   Edit the `LIMITATIONS` list to change content.

3. **Generative AI usage**: a structured block with colour-coded rows
   for what worked, what was harder, and what required care.
   Edit the `AI_BLOCKS` list to change content.

All three are injected by finding their `<h4>` anchor in the converted
markdown and replacing it with the styled HTML. If injection fails,
the warning tells you exactly which block to check in the QMD.

In [62]:
# ── Build conclusions section entirely in HTML ─────────────────────────────────

# ── RQ answers ────────────────────────────────────────────────────────────────
rq_html = (
    '<h2 id="what-this-means-for-zurich">6. What This Means for Zurich</h2>'
    '<blockquote><p>Three research questions, six models, 35,000 hours of data. '
    'Here is what we learned, and what EWZ can do with it.</p></blockquote>'

    '<h3>RQ1: What drives electricity demand?</h3>'
    '<p>Temperature and time of day are the primary drivers, and their effects interact. '
    'The GAM partial dependence plot shows the clearest picture: demand rises steeply below 5°C '
    '(heating regime), stays flat between 10 and 25°C (neutral regime), and increases again above 30°C '
    '(cooling regime). This J-shape explains why simple linear temperature terms underperform in winter '
    'and on summer heatwave days. The daily cycle (hour features) is the single strongest source of '
    'predictable variation, accounting for a larger share of explained variance than any weather '
    'variable individually.</p>'

    '<h3>RQ2: Can we predict peak hours?</h3>'
    '<blockquote><p>Peak hours are predictable from publicly available weather and time data alone '
    'with 96% AUC. EWZ could flag high-risk hours the evening before, giving operators time to act.</p></blockquote>'
    '<p>Yes, with high discriminative accuracy. The GLM Binomial reaches AUC 0.96 and 94% overall accuracy '
    'on the 2025 test set using only weather and time features. Peak hours are concentrated in cold winter '
    'weekday mornings, and the model has learned this structure cleanly. The main operational limitation '
    'is peak recall (65% at threshold 0.5), but this can be tuned by lowering the classification threshold. '
    'For a grid alarm application where a missed peak is more costly than a false alarm, a threshold '
    'around 0.3 would be more appropriate.</p>'

    '<h3>RQ3: Can we model the daily count of peak hours?</h3>'
    '<blockquote><p>Summer and winter peak risk are driven by opposite temperature effects '
    'a critical distinction for seasonal grid planning.</p></blockquote>'
    '<p>The Poisson GLM captures the seasonal trend with R² = 0.69. The most informative coefficient is '
    'the T:season interaction, which reverses sign between winter and summer: in winter, higher '
    'temperatures reduce expected peak hours (less heating needed), while in summer, higher temperatures '
    'increase them (cooling load from heat waves). This asymmetry has practical value for planning: '
    'summer peak risk is driven by hot spells, winter peak risk by cold ones, and the two regimes '
    'need different responses.</p>'
)

# ── Recommendations roadmap ────────────────────────────────────────────────────
RECOMMENDATIONS = [
    {
        'number':   '1',
        'priority': 'Deploy Now',
        'title':    'GLM Binomial as Peak-Hour Alert',
        'detail':   (
            'Flag high-probability peak hours the evening before using only weather forecasts '
            'and the calendar. AUC of 0.96 makes this operationally reliable today '
            'no additional data or infrastructure needed. Set the classification threshold '
            'to 0.3 to prioritise recall over precision for grid alarm applications.'
        ),
    },
    {
        'number':   '2',
        'priority': 'Production Ready',
        'title':    'SVM for Hourly Demand Forecasting',
        'detail':   (
            'The SVM achieves the lowest RMSE overall (881 kW, 1.2% of mean demand) '
            'and is the most accurate model for hourly demand forecasting. '
            'It uses lagged demand features alongside weather and time variables, '
            'making it suitable for a live forecasting pipeline where the previous '
            "hour's demand is always observable."
        ),
    },
    {
        'number':   '3',
        'priority': 'Strategic',
        'title':    'Separate Summer & Winter Peak Strategies',
        'detail':   (
            'Summer peaks are driven by heat waves and require cooling-load management. '
            'Winter peaks are driven by cold snaps and require heating-load management. '
            'These are structurally different regimes with opposite temperature '
            'sensitivities, a single operational response will not work for both.'
        ),
    },
]

roadmap_html = '<h3>Operational Recommendations</h3><div class="roadmap">'
for r in RECOMMENDATIONS:
    roadmap_html += (
        f'<div class="roadmap-step">'
        f'<div class="roadmap-header">'
        f'<div class="roadmap-number">{r["number"]}</div>'
        f'<div class="roadmap-meta">'
        f'<div class="roadmap-priority">{r["priority"]}</div>'
        f'<div class="roadmap-title">{r["title"]}</div>'
        f'</div></div>'
        f'<div class="roadmap-detail">{r["detail"]}</div>'
        f'</div>'
    )
roadmap_html += '</div>'

# ── Project Limitations ──────────────────
ICON_CAUTION = '<svg width="32" height="32" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M10.29 3.86L1.82 18a2 2 0 0 0 1.71 3h16.94a2 2 0 0 0 1.71-3L13.71 3.86a2 2 0 0 0-3.42 0z"/><line x1="12" y1="9" x2="12" y2="13"/><line x1="12" y1="17" x2="12.01" y2="17"/></svg>'

ICON_GRID    = '<svg width="32" height="32" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="3" y="3" width="7" height="7"/><rect x="14" y="3" width="7" height="7"/><rect x="14" y="14" width="7" height="7"/><rect x="3" y="14" width="7" height="7"/></svg>'

LIMITATIONS = [
    {
        'icon':  ICON_CAUTION,
        'title': 'Forecast Uncertainty',
        'text': (
            'The models assume the availability of perfect weather data. In real-world '
            'grid operations, errors in temperature or solar radiation forecasts would '
            'directly affect the accuracy of these demand predictions.'
        ),
    },
    {
        'icon':  ICON_GRID,
        'title': 'Technological Shifts',
        'text': (
            'The training data reflects 2022–2024 patterns. Sudden changes in Zurich’s '
            'energy mix, such as rapid adoption of heat pumps or electric vehicle '
            'fast-charging networks, may create new demand peaks not captured by trends.'
        ),
    },
]

lim_html = '<h3>Project Limitations</h3><div class="limitations">'
for l in LIMITATIONS:
    lim_html += (
        f'<div class="lim-card">'
        f'<span class="lim-icon">{l["icon"]}</span>'
        f'<div class="lim-title">{l["title"]}</div>'
        f'<div class="lim-text">{l["text"]}</div>'
        f'</div>'
    )
lim_html += '</div>'

# ── AI SVG icons ───────────────────────────────────────────────────────────────
ICON_CHECK = '<svg viewBox="0 0 20 20" fill="none" stroke="#4a7c59" stroke-width="2.5" stroke-linecap="round"><polyline points="4,10 8,14 16,6"/></svg>'
ICON_WAVE  = '<svg viewBox="0 0 20 20" fill="none" stroke="#4a7c59" stroke-width="2" stroke-linecap="round"><path d="M2,10 Q5,6 8,10 Q11,14 14,10 Q17,6 20,10"/></svg>'
ICON_EYE   = '<svg viewBox="0 0 20 20" fill="none" stroke="#4a7c59" stroke-width="2" stroke-linecap="round"><ellipse cx="10" cy="10" rx="8" ry="5"/><circle cx="10" cy="10" r="2.5" fill="#4a7c59"/></svg>'

AI_BLOCKS = [
    {
        'type':  'good',
        'icon':  ICON_CHECK,
        'label': 'What worked well',
        'text':  (
            'Generating boilerplate: imports, standard plot templates, the HTML report '
            'structure, and error message explanations. These tasks are well-defined and '
            'the outputs were usually correct on the first try, saving considerable time '
            'on repetitive setup.'
        ),
    },
    {
        'type':  'hard',
        'icon':  ICON_WAVE,
        'label': 'What was harder',
        'text':  (
            'Getting coefficient interpretations right without manual checking. The tool '
            'would sometimes produce plausible-sounding but subtly wrong explanations '
            'for example conflating the log-scale coefficient with the percentage change '
            'in the original scale. It also could not make modelling decisions: whether '
            'to include a given interaction, whether the overdispersion in the Poisson '
            "model was severe enough to switch to negative binomial, or whether the GAM's "
            'GCV-selected lambda was too smooth.'
        ),
    },
    {
        'type':  'careful',
        'icon':  ICON_EYE,
        'label': 'What required care',
        'text':  (
            'Outputs looked credible and were often syntactically correct code that ran '
            'without errors, but running without errors is not the same as doing the '
            'right thing. In one case the suggested SVR subsample code did not preserve '
            'temporal order in the indices, which we caught by reviewing the logic before '
            'running. We treated all AI output as a draft requiring verification against '
            'actual model outputs and a rewrite for specificity.'
        ),
    },
]

ai_html = (
    '<h2 id="how-we-used-generative-ai">7. How We Used Generative AI</h2>'
    '<p>We used a generative AI assistant throughout the project for several different purposes. '
    'On the code side, it helped with structuring notebooks, debugging, and working through '
    'methodological questions like why TimeSeriesSplit is the right approach for time series '
    'cross-validation rather than random KFold.</p>'
    '<div class="ai-section">'
)
for a in AI_BLOCKS:
    ai_html += (
        f'<div class="ai-block">'
        f'<div class="ai-signal {a["type"]}"></div>'
        f'<div class="ai-icon {a["type"]}">{a["icon"]}</div>'
        f'<div class="ai-content">'
        f'<div class="ai-label">{a["label"]}</div>'
        f'<div class="ai-text">{a["text"]}</div>'
        f'</div>'
        f'</div>'
    )
ai_html += '</div>'

# ── End of page wind turbine illustration ──────────────────────────────────────
footer_html = """
<div class="page-footer">
  <p class="footer-text">Powering Zurich's grid with data, one forecast at a time.</p>
  <svg viewBox="0 0 900 280" xmlns="http://www.w3.org/2000/svg" style="width:100%;display:block;">

    <!-- Sky gradient -->
    <defs>
      <linearGradient id="sky" x1="0" y1="0" x2="0" y2="1">
        <stop offset="0%" stop-color="#e8f0e8" stop-opacity="0"/>
        <stop offset="100%" stop-color="#4a7c59" stop-opacity="0.15"/>
      </linearGradient>
      <linearGradient id="hill" x1="0" y1="0" x2="0" y2="1">
        <stop offset="0%" stop-color="#4a7c59"/>
        <stop offset="100%" stop-color="#1a4a2e"/>
      </linearGradient>
    </defs>

    <!-- Rolling hills -->
    <ellipse cx="450" cy="320" rx="520" ry="100" fill="url(#hill)" opacity="0.25"/>
    <ellipse cx="150" cy="310" rx="280" ry="80" fill="url(#hill)" opacity="0.2"/>
    <ellipse cx="780" cy="315" rx="220" ry="70" fill="url(#hill)" opacity="0.2"/>

    <!-- Turbine 1 (left) -->
    <line x1="180" y1="260" x2="180" y2="140" stroke="#4a7c59" stroke-width="3" opacity="0.6"/>
    <circle cx="180" cy="140" r="5" fill="#4a7c59" opacity="0.7"/>
    <line x1="180" y1="140" x2="180" y2="85" stroke="#4a7c59" stroke-width="2.5" opacity="0.6" transform="rotate(-15 180 140)"/>
    <line x1="180" y1="140" x2="180" y2="85" stroke="#4a7c59" stroke-width="2.5" opacity="0.6" transform="rotate(105 180 140)"/>
    <line x1="180" y1="140" x2="180" y2="85" stroke="#4a7c59" stroke-width="2.5" opacity="0.6" transform="rotate(225 180 140)"/>

    <!-- Turbine 2 (centre) -->
    <line x1="450" y1="270" x2="450" y2="110" stroke="#1a4a2e" stroke-width="4" opacity="0.7"/>
    <circle cx="450" cy="110" r="7" fill="#1a4a2e" opacity="0.8"/>
    <line x1="450" y1="110" x2="450" y2="40" stroke="#1a4a2e" stroke-width="3" opacity="0.7" transform="rotate(0 450 110)"/>
    <line x1="450" y1="110" x2="450" y2="40" stroke="#1a4a2e" stroke-width="3" opacity="0.7" transform="rotate(120 450 110)"/>
    <line x1="450" y1="110" x2="450" y2="40" stroke="#1a4a2e" stroke-width="3" opacity="0.7" transform="rotate(240 450 110)"/>

    <!-- Turbine 3 (right) -->
    <line x1="720" y1="260" x2="720" y2="145" stroke="#4a7c59" stroke-width="3" opacity="0.6"/>
    <circle cx="720" cy="145" r="5" fill="#4a7c59" opacity="0.7"/>
    <line x1="720" y1="145" x2="720" y2="90" stroke="#4a7c59" stroke-width="2.5" opacity="0.6" transform="rotate(30 720 145)"/>
    <line x1="720" y1="145" x2="720" y2="90" stroke="#4a7c59" stroke-width="2.5" opacity="0.6" transform="rotate(150 720 145)"/>
    <line x1="720" y1="145" x2="720" y2="90" stroke="#4a7c59" stroke-width="2.5" opacity="0.6" transform="rotate(270 720 145)"/>

    <!-- Small turbine far left -->
    <line x1="60" y1="255" x2="60" y2="185" stroke="#7a9e87" stroke-width="2" opacity="0.5"/>
    <circle cx="60" cy="185" r="3.5" fill="#7a9e87" opacity="0.6"/>
    <line x1="60" y1="185" x2="60" y2="148" stroke="#7a9e87" stroke-width="1.8" opacity="0.5" transform="rotate(20 60 185)"/>
    <line x1="60" y1="185" x2="60" y2="148" stroke="#7a9e87" stroke-width="1.8" opacity="0.5" transform="rotate(140 60 185)"/>
    <line x1="60" y1="185" x2="60" y2="148" stroke="#7a9e87" stroke-width="1.8" opacity="0.5" transform="rotate(260 60 185)"/>

    <!-- Small turbine far right -->
    <line x1="850" y1="255" x2="850" y2="190" stroke="#7a9e87" stroke-width="2" opacity="0.5"/>
    <circle cx="850" cy="190" r="3.5" fill="#7a9e87" opacity="0.6"/>
    <line x1="850" y1="190" x2="850" y2="153" stroke="#7a9e87" stroke-width="1.8" opacity="0.5" transform="rotate(-10 850 190)"/>
    <line x1="850" y1="190" x2="850" y2="153" stroke="#7a9e87" stroke-width="1.8" opacity="0.5" transform="rotate(110 850 190)"/>
    <line x1="850" y1="190" x2="850" y2="153" stroke="#7a9e87" stroke-width="1.8" opacity="0.5" transform="rotate(230 850 190)"/>

    <!-- Ground line -->
    <path d="M0,265 Q225,240 450,255 Q675,270 900,255 L900,280 L0,280 Z" fill="#4a7c59" opacity="0.3"/>
    <path d="M0,272 Q225,255 450,265 Q675,275 900,265 L900,280 L0,280 Z" fill="#1a4a2e" opacity="0.25"/>

    <!-- Sun / data node -->
    <circle cx="820" cy="70" r="28" fill="#e8f0e8" opacity="0.5"/>
    <circle cx="820" cy="70" r="18" fill="#4a7c59" opacity="0.25"/>
    <circle cx="820" cy="70" r="8" fill="#4a7c59" opacity="0.4"/>

    <!-- Data flow lines connecting turbines to sun -->
    <path d="M180,140 Q400,60 812,70" stroke="#7a9e87" stroke-width="1" stroke-dasharray="4,6" fill="none" opacity="0.35"/>
    <path d="M450,110 Q630,50 812,70" stroke="#7a9e87" stroke-width="1" stroke-dasharray="4,6" fill="none" opacity="0.35"/>
    <path d="M720,145 Q770,100 812,70" stroke="#7a9e87" stroke-width="1" stroke-dasharray="4,6" fill="none" opacity="0.35"/>

  </svg>
</div>
"""

# ── Append everything to html_body ────────────────────────────────────────────
html_body += rq_html + roadmap_html + lim_html + ai_html + footer_html

# ── Add TOC entries for new sections ──────────────────────────────────────────
toc_items.append('<li><a href="#what-this-means-for-zurich">6. What This Means for Zurich</a></li>')
toc_items.append('<li><a href="#how-we-used-generative-ai">7. How We Used Generative AI</a></li>')
toc_html = (
    '<nav id="toc"><h2>Contents</h2><ul>'
    + ''.join(toc_items)
    + '</ul></nav>'
)

print('Conclusions appended  ✓')
print('Limitations appended  ✓')
print('AI section appended   ✓')
print('TOC updated           ✓')

Conclusions appended  ✓
Limitations appended  ✓
AI section appended   ✓
TOC updated           ✓


## Step 9: Assemble & Write

Combine all pieces into the final HTML file:
hero → metrics bar → TOC + content → close.

The output is a single self-contained file with no external dependencies.
File size is large because all figures are embedded as base64.

In [63]:
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Electricity Demand Forecasting for Zurich (EWZ)</title>
<style>{css}</style>
</head>
<body>

{hero}
{metrics_bar}

<div id="layout">
  {toc_html}
  <div id="content">
    {html_body}
  </div>
</div>

</body>
</html>"""

# ── Write file ─────────────────────────────────────────────────────────────────
with open(HTML_PATH, 'w', encoding='utf-8') as f:
    f.write(html)

size_mb = os.path.getsize(HTML_PATH) / 1024 / 1024
print(f'report.html written ✓  ({size_mb:.1f} MB)')
print(f'Sections: {len(toc_items)} · Images: {len(embedded)} · Missing: {len(missing)}')

report.html written ✓  (4.9 MB)
Sections: 25 · Images: 19 · Missing: 0
